## Boolean model - BoNesis

In [1]:
# === PARAMETERS ===
input_file = "/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Matrix/data_binarized.json"  # binarised files
patient = "P2"
output_directory = "/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Plots"

In [2]:
import sys
import json
import scanpy as sc
import numpy as np
import pandas as pd
import gc
import warnings
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import bonesis
import pyreadr

ipylab module is not installed, menus and toolbar are disabled.


Data binarized per macrostates

In [3]:
# data needs to be in dictionary type 
with open(input_file) as f:
    data = json.load(f)

Influence graph

In [4]:
result = pyreadr.read_r("/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Data/dorothea_hs.rda")
dorothea = result["dorothea_hs"]

# Keep the genes in confidence A, B or C 
dorothea = dorothea[dorothea["confidence"].isin(["A","B","C"])]

# Keep the genes that are in my data
genes = set()
for gene_data in data.values():
    genes.update(gene_data.keys())

dorothea_filter = dorothea[
    (dorothea['tf'].isin(genes)) | 
    (dorothea['target'].isin(genes))
].reset_index(drop=True)
print(f"Number of genes original DoRothEA : {len(dorothea)}")
print(f"Number of genes afetr filtering : {len(dorothea_filter)}")

# Adapt to the format required for BoNesis 
influences = []
for _, row in dorothea_filter.iterrows():
    tf = row["tf"]
    target = row["target"]
    sign = int(row["mor"])   # 1 or -1
    influences.append(
        (tf, target, dict(sign=sign)))
print(influences[:10])

Number of genes original DoRothEA : 13223
Number of genes afetr filtering : 9102
[('AHR', 'CYP1A1', {'sign': 1}), ('AHR', 'CYP1A2', {'sign': 1}), ('AHR', 'CYP1B1', {'sign': 1}), ('AHR', 'FOS', {'sign': 1}), ('AHR', 'MYC', {'sign': 1}), ('AHR', 'UGT1A6', {'sign': 1}), ('AHR', 'ASAP1', {'sign': 1}), ('AHR', 'ERG', {'sign': 1}), ('AHR', 'VGLL4', {'sign': 1}), ('AHR', 'ARHGAP15', {'sign': 1})]


In [5]:
influences = []

for _, row in dorothea_filter.iterrows():
    tf = row["tf"]
    target = row["target"]
    sign = int(row["mor"])   # 1 or -1
    influences.append(
        (tf, target, dict(sign=sign)))

print(influences[:10])

[('AHR', 'CYP1A1', {'sign': 1}), ('AHR', 'CYP1A2', {'sign': 1}), ('AHR', 'CYP1B1', {'sign': 1}), ('AHR', 'FOS', {'sign': 1}), ('AHR', 'MYC', {'sign': 1}), ('AHR', 'UGT1A6', {'sign': 1}), ('AHR', 'ASAP1', {'sign': 1}), ('AHR', 'ERG', {'sign': 1}), ('AHR', 'VGLL4', {'sign': 1}), ('AHR', 'ARHGAP15', {'sign': 1})]


In [6]:
dom1 = bonesis.InfluenceGraph(influences)
dom1

# computing graph layout...


KeyboardInterrupt: 

Constraints

In [ ]:
bo = bonesis.BoNesis(dom1, data)

In [ ]:
~bo.obs("I1") >= ~bo.obs("T1")
~bo.obs("I1") >= ~bo.obs("T3")
~bo.obs("I2") >= ~bo.obs("T2")
~bo.obs("I2") >= ~bo.obs("T3")

In [ ]:
for bn in bo.boolean_networks(limit=2): # limit is optional
    print(bn)

In [ ]:
# As dataframe 
solutions = list(bo.boolean_networks())
pd.DataFrame(solutions)